# ColdStart Killer — Insert 3K MVP Dataset into MongoDB

⚠️ Run only after:
1. Notebook 01 has generated `analysis/mvp_3000_items_diverse.csv`
2. MongoDB Atlas cluster is ready
3. `.env` is filled
4. Atlas Vector Search index and Atlas Search text index have been created manually

## 1. Setup imports

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.indexing import (
    build_contextual_header,
    build_hype_units,
    build_item_doc_from_mvp_row,
    build_proposition_units,
    estimate_indexing_size,
    index_items_from_dataframe,
)
from src.llm_hype import generate_hype_queries_llm
from src.llm_propositions import extract_propositions_llm
from src.embeddings import embed_texts
from src.mongodb import collection_counts, ping_mongodb
from src.validation import assert_required_columns

## 2. Load env and connect MongoDB

In [ ]:
ping_mongodb()

## 3. Load `analysis/mvp_3000_items_diverse.csv`

In [ ]:
csv_path = ROOT / "analysis" / "mvp_3000_items_diverse.csv"
df = pd.read_csv(csv_path)
df.shape

## 4. Validate required columns

In [ ]:
assert_required_columns(df)
"required columns OK"

## 5. Estimate indexing size

In [ ]:
estimate_indexing_size(df)

## 6. Dry run 3 items

In [ ]:
dry_run_3 = index_items_from_dataframe(df, limit=3, dry_run=True, sleep_seconds=0.5)
dry_run_3

## 7. Show sample item doc

In [ ]:
sample_item = build_item_doc_from_mvp_row(df.iloc[0])
sample_item

## 8. Show sample proposition units

In [ ]:
sample_propositions = extract_propositions_llm(sample_item)
sample_prop_units = build_proposition_units(sample_item, sample_propositions)
sample_prop_units[:3]

## 9. Show sample HyPE units

In [ ]:
sample_hype = generate_hype_queries_llm(sample_item, sample_propositions)
sample_embedding_texts = [f"{build_contextual_header(sample_item)} {query['raw_text']}" for query in sample_hype]
sample_embeddings = embed_texts(sample_embedding_texts)
sample_hype_units = build_hype_units(sample_item, sample_hype, sample_embeddings)
sample_hype_units[:3]

## 10. Insert 10 items

In [ ]:
insert_10 = index_items_from_dataframe(df, limit=10, dry_run=False, sleep_seconds=0.5)
insert_10

## 11. Count MongoDB collections

In [ ]:
collection_counts()

## 12. Insert 50 items

In [ ]:
insert_50 = index_items_from_dataframe(df, limit=50, dry_run=False, sleep_seconds=0.5)
insert_50

## 13. Optional larger inserts

In [ ]:
# insert_500 = index_items_from_dataframe(df, limit=500, dry_run=False, sleep_seconds=0.5)
# insert_1000 = index_items_from_dataframe(df, limit=1000, dry_run=False, sleep_seconds=0.5)
# insert_3000 = index_items_from_dataframe(df, limit=3000, dry_run=False, sleep_seconds=0.5)

## 14. Final stats

In [ ]:
{
    "counts": collection_counts(),
    "estimated_full_size": estimate_indexing_size(df),
}